# 12 - Spatial Math: SE(3) And Quaternions

## What / Why / How

**What we are trying to do:** Move from 2D frames into 3D rigid transforms, transform inverses, and quaternions.

**Why this matters:** Real robots operate in 3D. Arms, cameras, grippers, maps, and simulators all depend on reliable SE(3) reasoning.

**How we will do it:** Construct rotation matrices and homogeneous transforms, invert them, transform points, and combine simple quaternions.

## Goal

Robotics in 3D uses rotations, rigid transforms, and quaternions.

This notebook builds a practical SE(3) vocabulary:

- Rotation matrices
- Homogeneous transforms
- Quaternions
- Transform inversion and composition

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
from robotics_mastery.geometry import quaternion_from_axis_angle, quaternion_multiply

def rotx(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]], dtype=float)

def rotz(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=float)

def transform3(R, t):
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t
    return T

def invert_transform(T):
    R = T[:3, :3]
    t = T[:3, 3]
    T_inv = np.eye(4)
    T_inv[:3, :3] = R.T
    T_inv[:3, 3] = -R.T @ t
    return T_inv

T_world_camera = transform3(rotz(np.deg2rad(30)) @ rotx(np.deg2rad(10)), [1.0, 0.5, 1.2])
T_camera_world = invert_transform(T_world_camera)
print("T_world_camera:\n", T_world_camera)
print("inverse check:\n", T_camera_world @ T_world_camera)

In [ ]:
qz = quaternion_from_axis_angle(np.array([0, 0, 1]), np.deg2rad(90))
qx = quaternion_from_axis_angle(np.array([1, 0, 0]), np.deg2rad(30))
q_combined = quaternion_multiply(qz, qx)
print("qz:", qz)
print("qx:", qx)
print("combined:", q_combined)
print("unit norm:", np.linalg.norm(q_combined))

In [ ]:
point_camera = np.array([0.2, -0.1, 1.5, 1.0])
point_world = T_world_camera @ point_camera
roundtrip = T_camera_world @ point_world
print("world point:", point_world[:3])
print("roundtrip error:", np.linalg.norm(roundtrip - point_camera))

## Exercises

1. Compose three transforms: world to base, base to wrist, wrist to gripper.
2. Verify every rotation matrix has determinant 1.
3. Explain why quaternions are often preferred over Euler angles in robot software.